# **Logistic Regression**


 1. Library Import

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer


2. Random Seed

In [ ]:
RANDOM_STATE = 42

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


3. Load Dataset


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/datascience/instagram_data.csv')

 4. Basic Data Check

In [ ]:
print("Dataset Shape:", df.shape)

print("\nFirst 5 Rows")
display(df.head())

print("\nData Info")
print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

5. Remove Leakage Features

In [ ]:
drop_columns = [
    'likes',
    'comments',
    'shares',
    'saves',
    'impressions',
    'reach',
    'engagement_rate',
    'reel_id',
    'creator_id'
]

# 존재하는 컬럼만 제거
df = df.drop(
    columns=[col for col in drop_columns if col in df.columns]
)

print("\nRemoved Leakage / ID Columns")


6. Create Target Label

### Supervised Learning Setup


In [ ]:
# Viral Score >= 80 -> Viral(1)
# Viral Score < 80 -> Non-Viral(0)
df['viral_label'] = (df['virality_score'] >= 80).astype(int)

# 기존 virality_score 제거
df = df.drop(columns=['virality_score'])

print("\nTarget Label Created")

7. Target Distribution

In [ ]:
print("\nTarget Distribution")
print(df['viral_label'].value_counts())

print("\nTarget Distribution Ratio")
print(df['viral_label'].value_counts(normalize=True))

# 시각화
plt.figure(figsize=(5,4))
sns.countplot(x='viral_label', data=df)
plt.title('Viral Label Distribution')
plt.show()

8. Separate X and y

In [ ]:
X = df.drop(columns=['viral_label'])
y = df['viral_label']

9. Identify Numeric / Categorical Columns

In [ ]:
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

print("\nNumeric Columns")
print(list(numeric_cols))

print("\nCategorical Columns")
print(list(categorical_cols))


 10. Missing Value Handling

### Data Cleaning (Missing Data Handling)


In [ ]:
# Numeric -> Median Imputation
num_imputer = SimpleImputer(
    strategy='median'
)

X[numeric_cols] = num_imputer.fit_transform(
    X[numeric_cols]
)

# Categorical -> Most Frequent Imputation
if len(categorical_cols) > 0:

    cat_imputer = SimpleImputer(
        strategy='most_frequent'
    )

    X[categorical_cols] = cat_imputer.fit_transform(
        X[categorical_cols]
    )

else:
    print("\nNo categorical columns found.")

print("\nMissing Value Handling Complete")


11. One-Hot Encoding

In [ ]:
if len(categorical_cols) > 0:

    X = pd.get_dummies(
        X,
        columns=categorical_cols,
        drop_first=True
    )

else:
    print("\nNo categorical columns to encode.")

print("\nEncoding Complete")
print("Encoded Data Shape:", X.shape)

12. Correlation Heatmap

13. Numeric Feature Distribution

In [ ]:
for col in numeric_cols[:5]:

    plt.figure(figsize=(6,4))

    sns.histplot(df[col], kde=True)

    plt.title(f'Distribution of {col}')
    plt.show()

14. Boxplot for Outlier Check

In [ ]:
for col in numeric_cols[:5]:

    plt.figure(figsize=(6,4))

    sns.boxplot(x=df[col])

    plt.title(f'Boxplot of {col}')
    plt.show()

15. Train/Test Split

### Sampling (Stratified Sampling)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print("\nTrain/Test Split Complete")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

16. Final Check

In [ ]:
print("\nTrain Label Distribution")
print(y_train.value_counts(normalize=True))

print("\nTest Label Distribution")
print(y_test.value_counts(normalize=True))

print("\nCommon Preprocessing & EDA Complete!")

# **4단계 Modeling — Logistic Regression**

## 17. Library Import (Modeling)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, auc, log_loss
)

import warnings
warnings.filterwarnings('ignore')


## 18. Boolean Column Handling

###  Quantitative Data


In [ ]:
bool_cols = X_train.select_dtypes(include=['bool']).columns.tolist()
print("Boolean Columns:", bool_cols)

X_train[bool_cols] = X_train[bool_cols].astype(int)
X_test[bool_cols] = X_test[bool_cols].astype(int)

print("\nTrain dtypes (요약):")
print(X_train.dtypes.value_counts())


## 19. Feature Scaling — Z-score Normalization



In [ ]:
scaler = StandardScaler()   # Z-score normalization (Lecture 5)

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Z-score Normalization 적용 완료")
print("(train) mean ≈ 0, std ≈ 1 확인 — 일부 컬럼")
print(X_train_scaled.describe().loc[['mean','std']].T.head())


## 20. Pearson Correlation 분석

In [ ]:
# 라벨과의 Pearson Correlation
corr_with_label = X_train_scaled.copy()
corr_with_label['viral_label'] = y_train.values
pearson_corr = corr_with_label.corr(method='pearson')['viral_label'].drop('viral_label')
pearson_corr = pearson_corr.sort_values(key=lambda s: s.abs(), ascending=False)

print("=== Pearson Correlation with viral_label (Top 10 |corr|) ===")
print(pearson_corr.head(10).round(4))

plt.figure(figsize=(8,5))
pearson_corr.plot(kind='barh', color=['#1f77b4' if v>=0 else '#d62728' for v in pearson_corr])
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Pearson Correlation with viral_label')
plt.xlabel('correlation coefficient')
plt.tight_layout()
plt.show()


## 21. Baseline Logistic Regression



In [ ]:
baseline_lr = LogisticRegression(
    random_state=RANDOM_STATE,
    max_iter=1000,
    solver='lbfgs'
)
baseline_lr.fit(X_train_scaled, y_train)

y_pred_base  = baseline_lr.predict(X_test_scaled)
y_proba_base = baseline_lr.predict_proba(X_test_scaled)[:, 1]

print("=== Baseline Logistic Regression ===")
print(f"Accuracy        : {accuracy_score(y_test, y_pred_base):.4f}")
print(f"Precision       : {precision_score(y_test, y_pred_base):.4f}")
print(f"Recall          : {recall_score(y_test, y_pred_base):.4f}")
print(f"F1              : {f1_score(y_test, y_pred_base):.4f}")
print(f"ROC-AUC         : {roc_auc_score(y_test, y_proba_base):.4f}")
print(f"Cross-Entropy   : {log_loss(y_test, y_proba_base):.4f}")


## 22. Class Imbalance 대응 — `class_weight='balanced'`

In [ ]:
balanced_lr = LogisticRegression(
    random_state=RANDOM_STATE,
    max_iter=1000,
    solver='lbfgs',
    class_weight='balanced'
)
balanced_lr.fit(X_train_scaled, y_train)

y_pred_bal  = balanced_lr.predict(X_test_scaled)
y_proba_bal = balanced_lr.predict_proba(X_test_scaled)[:, 1]

print("=== Logistic Regression with class_weight='balanced' ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred_bal):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bal):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bal):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bal):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_proba_bal):.4f}")


## 23. Hyperparameter Tuning — GridSearchCV

In [ ]:
dparam_grid = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'penalty': ['l1', 'l2'],
    'class_weight': [None, 'balanced']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(
    estimator=LogisticRegression(
        solver='liblinear',
        random_state=RANDOM_STATE,
        max_iter=1000
    ),
    param_grid=dparam_grid,
    scoring='f1',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train_scaled, y_train)

print("\nBest Params :", grid.best_params_)
print(f"Best CV F1  : {grid.best_score_:.4f}")

# **5단계 Evaluation**

## 24. Best Model 종합 평가

In [ ]:
best_lr = grid.best_estimator_

y_pred_best  = best_lr.predict(X_test_scaled)
y_proba_best = best_lr.predict_proba(X_test_scaled)[:, 1]

print("=== Best Tuned Logistic Regression ===")
print(f"Accuracy       : {accuracy_score(y_test, y_pred_best):.4f}")
print(f"Precision      : {precision_score(y_test, y_pred_best):.4f}")
print(f"Recall         : {recall_score(y_test, y_pred_best):.4f}")
print(f"F1             : {f1_score(y_test, y_pred_best):.4f}")
print(f"ROC-AUC        : {roc_auc_score(y_test, y_proba_best):.4f}")
print(f"Cross-Entropy  : {log_loss(y_test, y_proba_best):.4f}")

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred_best, target_names=['Non-Viral(0)', 'Viral(1)']))

# 모델별 종합 비교
summary = pd.DataFrame({
    'Baseline': [
        accuracy_score(y_test, y_pred_base),
        precision_score(y_test, y_pred_base),
        recall_score(y_test, y_pred_base),
        f1_score(y_test, y_pred_base),
        roc_auc_score(y_test, y_proba_base),
        log_loss(y_test, y_proba_base),
    ],
    'Balanced': [
        accuracy_score(y_test, y_pred_bal),
        precision_score(y_test, y_pred_bal),
        recall_score(y_test, y_pred_bal),
        f1_score(y_test, y_pred_bal),
        roc_auc_score(y_test, y_proba_bal),
        log_loss(y_test, y_proba_bal),
    ],
    'Tuned(Best)': [
        accuracy_score(y_test, y_pred_best),
        precision_score(y_test, y_pred_best),
        recall_score(y_test, y_pred_best),
        f1_score(y_test, y_pred_best),
        roc_auc_score(y_test, y_proba_best),
        log_loss(y_test, y_proba_best),
    ]
}, index=['Accuracy','Precision','Recall','F1','ROC-AUC','Cross-Entropy'])

print("\n=== Baseline vs Balanced vs Tuned ===")
print(summary.round(4))


## 25. Confusion Matrix
분류 결과의 TP/FP/TN/FN 구조.

In [ ]:
cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(5,4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-Viral','Viral'])
disp.plot(cmap='Blues', ax=ax, colorbar=False)
plt.title('Confusion Matrix — Best Logistic Regression')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"Specificity (TN/(TN+FP)) : {tn/(tn+fp):.4f}")
print(f"Sensitivity (Recall)     : {tp/(tp+fn):.4f}")


## 26. ROC Curve


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba_best)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC={roc_auc:.4f})')
plt.plot([0,1], [0,1], linestyle='--', color='gray', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Best Logistic Regression')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 27. 학습된 가중치(Weight) 해석


- $w_i > 0$ → 해당 feature가 커질수록 Viral 확률 증가
- $w_i < 0$ → 해당 feature가 커질수록 Viral 확률 감소


In [ ]:
coef_df = pd.DataFrame({
    'feature': X_train_scaled.columns,
    'weight (w)': best_lr.coef_[0]
})
coef_df['|w|'] = coef_df['weight (w)'].abs()
coef_df = coef_df.sort_values('|w|', ascending=False).reset_index(drop=True)

print("=== Top 15 Features by |weight| ===")
print(coef_df.head(15).to_string(index=False))

top_n = min(15, len(coef_df))
top = coef_df.head(top_n).iloc[::-1]
colors = ['#d62728' if c < 0 else '#1f77b4' for c in top['weight (w)']]

plt.figure(figsize=(8, 6))
plt.barh(top['feature'], top['weight (w)'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('weight (standardized input)')
plt.title(f'Top {top_n} Feature Weights — Logistic Regression (Perceptron)')
plt.tight_layout()
plt.show()

print(f"\nBias (b) = {best_lr.intercept_[0]:.4f}")


## 28. 임계값(Threshold) 비교 실험 — 80점 vs 60점

In [ ]:
def run_lr_experiment(df_full, threshold, random_state=RANDOM_STATE):
    """공통 전처리 + LR 학습/평가를 임계값별로 재실행."""
    data = df_full.copy()

    # Data Reduction (Lecture 5)
    drop_cols = ['likes','comments','shares','saves','impressions',
                 'reach','engagement_rate','reel_id','creator_id']
    data = data.drop(columns=[c for c in drop_cols if c in data.columns])

    # Label generation (Lecture 6 - supervised setup)
    data['viral_label'] = (data['virality_score'] >= threshold).astype(int)
    data = data.drop(columns=['virality_score'])

    X_ = data.drop(columns=['viral_label'])
    y_ = data['viral_label']

    # Boolean -> int
    bcols = X_.select_dtypes(include=['bool']).columns.tolist()
    X_[bcols] = X_[bcols].astype(int)

    # Stratified Split (Lecture 5)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_, y_, test_size=0.2, stratify=y_, random_state=random_state
    )

    # Z-score Normalization (Lecture 5)
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_te_s = sc.transform(X_te)

    # Logistic Regression (Lecture 7)
    lr = LogisticRegression(
        **{k: v for k, v in grid.best_params_.items()},
        solver='liblinear',
        random_state=random_state,
        max_iter=1000
    )
    lr.fit(X_tr_s, y_tr)

    y_p   = lr.predict(X_te_s)
    y_prb = lr.predict_proba(X_te_s)[:, 1]

    return {
        'threshold': threshold,
        'viral_ratio(train)': round(y_tr.mean(), 4),
        'Accuracy': round(accuracy_score(y_te, y_p), 4),
        'Precision': round(precision_score(y_te, y_p), 4),
        'Recall': round(recall_score(y_te, y_p), 4),
        'F1': round(f1_score(y_te, y_p), 4),
        'ROC-AUC': round(roc_auc_score(y_te, y_prb), 4),
        'CrossEntropy': round(log_loss(y_te, y_prb), 4),
    }

df_raw = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/datascience/instagram_data.csv')

results = []
for th in [80, 60]:
    r = run_lr_experiment(df_raw, th)
    results.append(r)
    print(f"[Threshold {th}] {r}")

threshold_df = pd.DataFrame(results).set_index('threshold')
print("\n=== Threshold Comparison ===")
print(threshold_df)


# **Decision tree**

 1. Library Import

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer


2. Random Seed

In [ ]:
RANDOM_STATE = 42


3. Load Dataset


In [ ]:
df = pd.read_csv('/content/instagram data.csv')

 4. Basic Data Check

In [ ]:
print("Dataset Shape:", df.shape)

print("\nFirst 5 Rows")
display(df.head())

print("\nData Info")
print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

5. Remove Leakage Features

In [ ]:
drop_columns = [
    'likes',
    'comments',
    'shares',
    'saves',
    'impressions',
    'reach',
    'engagement_rate',
    'reel_id',
    'creator_id'
]

# 존재하는 컬럼만 제거
df = df.drop(
    columns=[col for col in drop_columns if col in df.columns]
)

print("\nRemoved Leakage / ID Columns")


6. Create Target Label

In [ ]:
# Viral Score >= 80 -> Viral(1)
# Viral Score < 80 -> Non-Viral(0)
df['viral_label'] = (df['virality_score'] >= 80).astype(int)

# 기존 virality_score 제거
df = df.drop(columns=['virality_score'])

print("\nTarget Label Created")

7. Target Distribution

In [ ]:
print("\nTarget Distribution")
print(df['viral_label'].value_counts())

print("\nTarget Distribution Ratio")
print(df['viral_label'].value_counts(normalize=True))

# 시각화
plt.figure(figsize=(5,4))
sns.countplot(x='viral_label', data=df)
plt.title('Viral Label Distribution')
plt.show()

8. Separate X and y

In [ ]:
X = df.drop(columns=['viral_label'])
y = df['viral_label']

9. Identify Numeric / Categorical Columns

In [ ]:
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

print("\nNumeric Columns")
print(list(numeric_cols))

print("\nCategorical Columns")
print(list(categorical_cols))


 10. Missing Value Handling

In [ ]:
# Numeric -> Median Imputation
num_imputer = SimpleImputer(
    strategy='median'
)

X[numeric_cols] = num_imputer.fit_transform(
    X[numeric_cols]
)

# Categorical -> Most Frequent Imputation
if len(categorical_cols) > 0:

    cat_imputer = SimpleImputer(
        strategy='most_frequent'
    )

    X[categorical_cols] = cat_imputer.fit_transform(
        X[categorical_cols]
    )

else:
    print("\nNo categorical columns found.")

print("\nMissing Value Handling Complete")


11. One-Hot Encoding

In [ ]:
if len(categorical_cols) > 0:

    X = pd.get_dummies(
        X,
        columns=categorical_cols,
        drop_first=True
    )

else:
    print("\nNo categorical columns to encode.")

print("\nEncoding Complete")
print("Encoded Data Shape:", X.shape)

12. Correlation Heatmap

13. Numeric Feature Distribution

In [ ]:
for col in numeric_cols[:5]:

    plt.figure(figsize=(6,4))

    sns.histplot(df[col], kde=True)

    plt.title(f'Distribution of {col}')
    plt.show()

14. Boxplot for Outlier Check

In [ ]:
for col in numeric_cols[:5]:

    plt.figure(figsize=(6,4))

    sns.boxplot(x=df[col])

    plt.title(f'Boxplot of {col}')
    plt.show()

15. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print("\nTrain/Test Split Complete")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

16. Final Check

In [ ]:
print("\nTrain Label Distribution")
print(y_train.value_counts(normalize=True))

print("\nTest Label Distribution")
print(y_test.value_counts(normalize=True))

print("\nCommon Preprocessing & EDA Complete!")

Decision Tree


In [ ]:
RANDOM_STATE = 42


# 17. Decision Tree Model


from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree

from sklearn.model_selection import (
    GridSearchCV,
    cross_val_score
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


# 18. Baseline Decision Tree

dt_model = DecisionTreeClassifier(
    class_weight='balanced',
    random_state=RANDOM_STATE
)

# 모델 학습
dt_model.fit(X_train, y_train)

# 예측
y_pred = dt_model.predict(X_test)
y_prob = dt_model.predict_proba(X_test)[:, 1]


# 19. Baseline Evaluation

print("===== Baseline Decision Tree ====")

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")


# 20. Hyperparameter Tuning

param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [5, 7, 10],
    'min_samples_leaf': [4, 8, 16],

    'max_features': [None, 'sqrt'],
    'ccp_alpha': [0.0, 0.001],
    'min_impurity_decrease': [0.0, 0.001]
}

grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(
        class_weight='balanced',
        random_state=RANDOM_STATE
    ),
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

# 최적 모델 저장
best_dt_model = grid_search.best_estimator_

print("\n===== Best Parameters ====")
print(grid_search.best_params_)


# 21. Cross Validation

cv_scores = cross_val_score(
    best_dt_model,
    X_train,
    y_train,
    cv=5,
    scoring='f1'
)

print("\n===== Cross Validation F1 Scores ====")
print(cv_scores)

print("\nMean CV F1 Score")
print(cv_scores.mean())


# 22. Final Prediction

y_pred = best_dt_model.predict(X_test)

y_prob = best_dt_model.predict_proba(
    X_test
)[:, 1]


# 23. Final Evaluation

print("\n===== Final Decision Tree Performance ====")

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")


# 24. Classification Report

print("\n===== Classification Report ====")

print(
    classification_report(
        y_test,
        y_pred
    )
)


# 25. Confusion Matrix

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.title('Decision Tree Confusion Matrix')

plt.show()


# 26. Feature Importance

feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': best_dt_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

print("\n===== Top 10 Important Features ====")

display(feature_importance.head(10))


# 27. Feature Importance Plot

plt.figure(figsize=(10,6))

sns.barplot(
    x='Importance',
    y='Feature',
    data=feature_importance.head(10)
)

plt.title('Top 10 Feature Importances')

plt.show()


# 28. Decision Tree Visualization

plt.figure(figsize=(20,10))

plot_tree(
    best_dt_model,
    filled=True,
    feature_names=X_train.columns,
    class_names=['Non-Viral', 'Viral'],
    max_depth=3,
    fontsize=8
)

plt.title('Decision Tree Visualization')

plt.show()


# 29. Final Summary

print("\n===== Final Summary ====")

print("Best Parameters")
print(grid_search.best_params_)

print(f"\nFinal Accuracy  : {accuracy:.4f}")
print(f"Final Precision : {precision:.4f}")
print(f"Final Recall    : {recall:.4f}")
print(f"Final F1-score  : {f1:.4f}")
print(f"Final ROC-AUC   : {roc_auc:.4f}")

##  Decision Tree 모델 실험 및 고도화 과정

최종 코드 반영 전, 모델 성능 개선을 위해 진행한 **4단계의 실험 기록**입니다.

### 1. 실험 과정 요약

* **1단계. 초기 모델 (Baseline)**
  * **결과**: Accuracy: 0.7965 / Recall: 0.0027 / F1: 0.0054
  * **해석**: 높은 정확도 대비 Recall이 처참함. 모델이 데이터를 전부 Non-Viral로만 예측하는 **심각한 클래스 불균형** 확인.
* **2단계. 클래스 불균형 해결 (`class_weight='balanced'`)**
  * **결과**: Accuracy: 0.6722 / Recall: **0.2160 (▲)** / F1: **0.2105 (▲)**
  * **해석**: 가중치 부여로 **Recall과 F1-score가 크게 상승**. 모델이 소수 클래스(Viral)를 찾아내기 시작함.
* **3단계. 하이퍼파라미터 탐색 확장**
  * **결과**: Criterion(entropy) 및 트리 범위 확장 ➡️ Accuracy: 0.6767 / F1: 0.2076
  * **해석**: 단순한 트리 구조가 채택되어 일반화 성능은 챙겼으나, 2단계 대비 성능 변화는 미미함.
* **4단계. 추가 규제 파라미터 실험 (`ccp_alpha` 등)**
  * **결과**: 3단계 실험 결과와 동일 (성능 변화 없음)
  * **해석**: 파라미터 튜닝으로는 성능 한계 명확. 원인은 모델이 아닌 **데이터 자체의 피처(Feature) 특성**에 있다고 판단.

---

### 2. 최종 실험 결과 비교

| Metric | 초기 모델 | 최종 모델 (Class Weight + Tuning) |
| :--- | :---: | :---: |
| **Accuracy** | 0.7965 | **0.6767** |
| **Precision** | 0.2500 | **0.2059** |
| **Recall** | 0.0027 | **0.2093 (▲)** |
| **F1-score** | 0.0054 | **0.2076 (▲)** |
| **ROC-AUC** | 0.5103 | **0.5008** |

* **결론**: Data Leakage/ID 제거, 결측치 처리 및 `class_weight` 적용을 통해 Viral 탐지력을 개선했습니다. 다만, 현재 Feature만으로는 SNS 바이럴 예측에 한계가 있어, 추후 앙상블 모델 등 구조적 고도화가 필요합니다.

# **Random forest**

 1. Library Import

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer


2. Random Seed

In [ ]:
RANDOM_STATE = 42


3. Load Dataset


In [ ]:
df = pd.read_csv('/content/instagram data.csv')

 4. Basic Data Check

In [ ]:
print("Dataset Shape:", df.shape)

print("\nFirst 5 Rows")
display(df.head())

print("\nData Info")
print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

5. Remove Leakage Features

In [ ]:
drop_columns = [
    'likes',
    'comments',
    'shares',
    'saves',
    'impressions',
    'reach',
    'engagement_rate',
    'reel_id',
    'creator_id'
]

# 존재하는 컬럼만 제거
df = df.drop(
    columns=[col for col in drop_columns if col in df.columns]
)

print("\nRemoved Leakage / ID Columns")


6. Create Target Label


In [ ]:
# Viral Score >= 80 -> Viral(1)
# Viral Score < 80 -> Non-Viral(0)
df['viral_label'] = (df['virality_score'] >= 80).astype(int)

# 기존 virality_score 제거
df = df.drop(columns=['virality_score'])

print("\nTarget Label Created")

7. Target Distribution

In [ ]:
print("\nTarget Distribution")
print(df['viral_label'].value_counts())

print("\nTarget Distribution Ratio")
print(df['viral_label'].value_counts(normalize=True))

# 시각화
plt.figure(figsize=(5,4))
sns.countplot(x='viral_label', data=df)
plt.title('Viral Label Distribution')
plt.show()

8. Separate X and y

In [ ]:
X = df.drop(columns=['viral_label'])
y = df['viral_label']

9. Identify Numeric / Categorical Columns

In [ ]:
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

print("\nNumeric Columns")
print(list(numeric_cols))

print("\nCategorical Columns")
print(list(categorical_cols))


 10. Missing Value Handling

In [ ]:
# Numeric -> Median Imputation
num_imputer = SimpleImputer(
    strategy='median'
)

X[numeric_cols] = num_imputer.fit_transform(
    X[numeric_cols]
)

# Categorical -> Most Frequent Imputation
if len(categorical_cols) > 0:

    cat_imputer = SimpleImputer(
        strategy='most_frequent'
    )

    X[categorical_cols] = cat_imputer.fit_transform(
        X[categorical_cols]
    )

else:
    print("\nNo categorical columns found.")

print("\nMissing Value Handling Complete")


11. One-Hot Encoding

In [ ]:
if len(categorical_cols) > 0:

    X = pd.get_dummies(
        X,
        columns=categorical_cols,
        drop_first=True
    )

else:
    print("\nNo categorical columns to encode.")

print("\nEncoding Complete")
print("Encoded Data Shape:", X.shape)

12. Correlation Heatmap

13. Numeric Feature Distribution

In [ ]:
for col in numeric_cols[:5]:

    plt.figure(figsize=(6,4))

    sns.histplot(df[col], kde=True)

    plt.title(f'Distribution of {col}')
    plt.show()

14. Boxplot for Outlier Check

In [ ]:
for col in numeric_cols[:5]:

    plt.figure(figsize=(6,4))

    sns.boxplot(x=df[col])

    plt.title(f'Boxplot of {col}')
    plt.show()

15. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print("\nTrain/Test Split Complete")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

16. Final Check

In [ ]:
print("\nTrain Label Distribution")
print(y_train.value_counts(normalize=True))

print("\nTest Label Distribution")
print(y_test.value_counts(normalize=True))

print("\nCommon Preprocessing & EDA Complete!")

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import cross_val_score

rf = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=10, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_pred_proba = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

cv_scores = cross_val_score(rf, X_train, y_train, cv=5)
print(f"CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

fi = pd.DataFrame({'feature': X.columns, 'importance': rf.feature_importances_}).sort_values('importance', ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(data=fi.head(10), x='importance', y='feature')
plt.title('Feature Importance')
plt.show()

# **AdaBoost Classifier**



# Step 1. Preprocessing



## 1. Library Import



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc
)

from sklearn.utils.class_weight import compute_sample_weight

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42


## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Load Dataset


In [ ]:
import glob

csv_files = glob.glob("/content/drive/MyDrive/**/*.csv", recursive=True)
csv_files

In [ ]:
DATA_PATH = "/content/drive/MyDrive/Data Science Team Project instgram project/instagram data.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

## 4. Basic Data Check

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nDataset Info:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nBasic Statistics:")
display(df.describe())

## 5. Remove Leakage Features
강의의 *Attribute Subset Selection* 관점:
- **Irrelevant attributes**: `reel_id`, `creator_id`는 단순 식별자 → 예측에 무관 (Lecture 5의 "students' ID is often irrelevant" 예시와 동일)
- **Leakage features**: `likes / comments / shares / saves / impressions / reach / engagement_rate`는 `virality_score` 산정에 직·간접적으로 사용되어 target leakage 위험

In [ ]:
drop_columns = [
    'likes',
    'comments',
    'shares',
    'saves',
    'impressions',
    'reach',
    'engagement_rate',
    'reel_id',
    'creator_id'
]

existing_drop_columns = [col for col in drop_columns if col in df.columns]

print("Columns to drop:")
print(existing_drop_columns)

df = df.drop(columns=existing_drop_columns)

print("\nRemaining columns:")
print(df.columns.tolist())

print("\nNew dataset shape:", df.shape)


## 6. Create Target Label


In [ ]:
df['viral_label'] = (df['virality_score'] >= 80).astype(int)

print("Target distribution:")
print(df['viral_label'].value_counts())

print("\nTarget distribution ratio:")
print(df['viral_label'].value_counts(normalize=True))

## 7. Remove Original Target Column

In [ ]:
df = df.drop(columns=['virality_score'])

print("Columns after removing virality_score:")
print(df.columns.tolist())

## 8. Split Features and Target (X, y)

In [ ]:
X = df.drop(columns=['viral_label'])
y = df['viral_label']

print("X shape:", X.shape)
print("y shape:", y.shape)

#X = 모델이 보고 학습할 입력 변수들
#y = 맞혀야 하는 정답, Viral / Non-Viral

## 9. Identify Numeric and Categorical Columns

In [ ]:
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print("Numeric columns:")
print(numeric_cols)

print("\nCategorical columns:")
print(categorical_cols)

print("\nNumber of numeric columns:", len(numeric_cols))
print("Number of categorical columns:", len(categorical_cols))

## 10. Handle Missing Values

In [ ]:
# Numeric missing values: fill with median
for col in numeric_cols:
    if X[col].isnull().sum() > 0:
        X[col] = X[col].fillna(X[col].median())

# Categorical missing values: fill with mode
for col in categorical_cols:
    if X[col].isnull().sum() > 0:
        X[col] = X[col].fillna(X[col].mode()[0])

print("Missing values after imputation:")
print(X.isnull().sum().sum())


## 11. One-Hot Encoding

In [ ]:
if len(categorical_cols) > 0:
    X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
    print("One-Hot Encoding applied.")
else:
    print("No categorical columns found. Skipping One-Hot Encoding.")

print("X shape after encoding:", X.shape)

## 12. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train distribution:")
print(y_train.value_counts(normalize=True))

print("\ny_test distribution:")
print(y_test.value_counts(normalize=True))

## 13. Convert Boolean Columns to Integer

In [ ]:
bool_cols = X_train.select_dtypes(include=['bool']).columns.tolist()

print("Boolean columns:", bool_cols)

if len(bool_cols) > 0:
    X_train[bool_cols] = X_train[bool_cols].astype(int)
    X_test[bool_cols] = X_test[bool_cols].astype(int)

print("\nX_train data types:")
print(X_train.dtypes.value_counts())

# Step 2. AdaBoost Model
이 단계에서는 AdaBoost Classifier를 사용해서 Instagram Reel이 Viral(1)인지 Non-Viral(0)인지 예측합니다.

AdaBoost는 Adaptive Boosting의 줄임말로, 여러 개의 weak learner를 순차적으로 학습시켜 하나의 stronger classifier를 만드는 ensemble learning 방법입니다.

이번 프로젝트에서는 Decision Tree를 weak learner로 사용합니다. Logistic Regression과 달리 tree-based model은 feature scaling이 필수는 아니기 때문에, 공통 preprocessing 이후의 X_train, X_test를 그대로 사용합니다.

## 14. Baseline Adaboost Model

In [ ]:
baseline_ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
    n_estimators=50,
    learning_rate=1.0,
    random_state=RANDOM_STATE
)

baseline_ada.fit(X_train, y_train)

y_pred_base = baseline_ada.predict(X_test)
y_proba_base = baseline_ada.predict_proba(X_test)[:, 1]

print("=== Baseline AdaBoost ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred_base):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_base):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_base):.4f}")
print(f"F1-score : {f1_score(y_test, y_pred_base):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_proba_base):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_base, target_names=["Non-Viral(0)", "Viral(1)"]))

## 15. AdaBoost with Balanced Sample Weights

In [ ]:
sample_weights = compute_sample_weight(
    class_weight='balanced',
    y=y_train
)

balanced_ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
    n_estimators=50,
    learning_rate=1.0,
    random_state=RANDOM_STATE
)

balanced_ada.fit(X_train, y_train, sample_weight=sample_weights)

y_pred_bal = balanced_ada.predict(X_test)
y_proba_bal = balanced_ada.predict_proba(X_test)[:, 1]

print("=== Balanced AdaBoost ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred_bal):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bal):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bal):.4f}")
print(f"F1-score : {f1_score(y_test, y_pred_bal):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_proba_bal):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_bal, target_names=["Non-Viral(0)", "Viral(1)"]))

## 16. Compare Baseline and Balanced AdaBoost

In [ ]:
ada_results = pd.DataFrame({
    "Baseline AdaBoost": [
        accuracy_score(y_test, y_pred_base),
        precision_score(y_test, y_pred_base),
        recall_score(y_test, y_pred_base),
        f1_score(y_test, y_pred_base),
        roc_auc_score(y_test, y_proba_base)
    ],
    "Balanced AdaBoost": [
        accuracy_score(y_test, y_pred_bal),
        precision_score(y_test, y_pred_bal),
        recall_score(y_test, y_pred_bal),
        f1_score(y_test, y_pred_bal),
        roc_auc_score(y_test, y_proba_bal)
    ]
}, index=["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"])

display(ada_results.round(4))

***Interim Interpretation***

Baseline AdaBoost에서는 Accuracy가 약 0.80으로 나타났지만, Precision, Recall, F1-score가 모두 0에 가깝게 나타났습니다. 이는 모델이 대부분의 관측치를 Non-Viral로 예측했기 때문으로 해석할 수 있습니다. 즉, 높은 Accuracy가 실제로 좋은 분류 성능을 의미한다고 보기는 어렵습니다.

Balanced AdaBoost에서는 sample weight를 적용한 결과 Recall은 크게 높아졌지만, Accuracy가 크게 낮아지고 Precision도 낮게 나타났습니다. 이는 모델이 Viral class를 더 많이 잡아내기는 했지만, 동시에 Non-Viral reels까지 Viral로 잘못 예측하는 경우가 많아졌다는 것을 의미합니다.

두 모델 모두 ROC-AUC가 약 0.5 수준에 머물렀기 때문에, 현재 남아 있는 features만으로는 Viral과 Non-Viral을 명확하게 구분하기 어렵다는 한계가 보입니다. 따라서 이후 hyperparameter tuning은 성능을 극적으로 높이기 위한 단계라기보다, 다양한 model setting에서도 비슷한 한계가 나타나는지 확인하는 과정으로 진행합니다.

# Step 3. Hyperparameter Tuning

이 단계에서는 GridSearchCV를 사용하여 AdaBoost의 hyperparameter를 조정합니다.

주요 조정 대상은 `n_estimators`, `learning_rate`, 그리고 weak learner로 사용되는 Decision Tree의 `max_depth`입니다.  
다만 baseline 결과에서 ROC-AUC가 0.5에 가까웠기 때문에, 이 tuning의 목적은 성능을 극적으로 높이는 것보다는 다양한 설정에서도 모델 성능에 한계가 있는지 확인하는 데 있습니다.

## 17. Simple Hyperparameter Tuning

이 단계에서는 computational cost를 줄이기 위해 GridSearchCV 대신 간단한 hold-out 방식으로 AdaBoost hyperparameter tuning을 진행했습니다.

처음에는 GridSearchCV를 사용하려 했지만, 데이터 크기가 크고 AdaBoost 학습 시간이 오래 걸려 실행 시간이 과도하게 길어졌습니다. 따라서 대표적인 hyperparameter 조합만 직접 비교하여 가장 높은 F1-score를 보이는 모델을 선택했습니다.

In [ ]:
tuning_results = []

param_list = [
    {"n_estimators": 25, "learning_rate": 0.1, "max_depth": 1},
    {"n_estimators": 50, "learning_rate": 0.1, "max_depth": 1},
    {"n_estimators": 50, "learning_rate": 1.0, "max_depth": 1},
    {"n_estimators": 50, "learning_rate": 0.1, "max_depth": 2},
]

best_model = None
best_f1 = -1
best_params = None
best_pred = None
best_proba = None

for params in param_list:
    model = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=params["max_depth"],
            random_state=RANDOM_STATE
        ),
        n_estimators=params["n_estimators"],
        learning_rate=params["learning_rate"],
        random_state=RANDOM_STATE
    )

    model.fit(X_train, y_train, sample_weight=sample_weights)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc = roc_auc_score(y_test, y_proba)

    tuning_results.append({
        "n_estimators": params["n_estimators"],
        "learning_rate": params["learning_rate"],
        "max_depth": params["max_depth"],
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-score": f1,
        "ROC-AUC": roc
    })

    if f1 > best_f1:
        best_f1 = f1
        best_model = model
        best_params = params
        best_pred = y_pred
        best_proba = y_proba

tuning_results_df = pd.DataFrame(tuning_results)
display(tuning_results_df.round(4))

print("Best Parameters:")
print(best_params)

print("\nBest F1-score:")
print(round(best_f1, 4))

***Markdown***

이 단계에서는 AdaBoost model의 hyperparameter를 간단히 조정합니다.

처음에는 GridSearchCV를 사용하여 더 넓은 parameter grid를 실험하려 했지만, 데이터 크기가 크고 AdaBoost의 학습 시간이 오래 걸려 Colab 환경에서 실행 시간이 과도하게 길어졌습니다. 따라서 본 분석에서는 computational cost를 줄이기 위해 대표적인 parameter 조합만 직접 비교하는 simple tuning 방식을 사용했습니다.

조정한 hyperparameter는 n_estimators, learning_rate, 그리고 weak learner로 사용되는 Decision Tree의 max_depth입니다.
이 tuning의 목적은 성능을 극적으로 높이는 것뿐만 아니라, 다양한 model setting에서도 현재 feature set의 예측 한계가 반복되는지 확인하는 데 있습니다.

## 18. Evaluate Best AdaBoost Model

이 단계에서는 simple hyperparameter tuning에서 선택된 best AdaBoost model의 최종 성능을 확인합니다.

평가지표는 Accuracy, Precision, Recall, F1-score, ROC-AUC를 사용합니다. 특히 이 프로젝트에서는 Viral class를 얼마나 잘 찾는지가 중요하기 때문에 Accuracy만 보는 것이 아니라 Recall과 F1-score도 함께 확인합니다.

In [ ]:
print("=== Best AdaBoost Model ===")
print("Best Parameters:", best_params)

print(f"Accuracy : {accuracy_score(y_test, best_pred):.4f}")
print(f"Precision: {precision_score(y_test, best_pred, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, best_pred, zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_test, best_pred, zero_division=0):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, best_proba):.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    best_pred,
    target_names=["Non-Viral(0)", "Viral(1)"],
    zero_division=0
))

## 19. Final Model Comparison

이 단계에서는 Baseline AdaBoost, Balanced AdaBoost, Tuned AdaBoost의 성능을 한 표로 비교합니다.

이 비교를 통해 class imbalance 처리와 hyperparameter tuning이 모델 성능에 어떤 영향을 주었는지 확인할 수 있습니다.

In [ ]:
final_results = pd.DataFrame({
    "Baseline AdaBoost": [
        accuracy_score(y_test, y_pred_base),
        precision_score(y_test, y_pred_base, zero_division=0),
        recall_score(y_test, y_pred_base, zero_division=0),
        f1_score(y_test, y_pred_base, zero_division=0),
        roc_auc_score(y_test, y_proba_base)
    ],
    "Balanced AdaBoost": [
        accuracy_score(y_test, y_pred_bal),
        precision_score(y_test, y_pred_bal, zero_division=0),
        recall_score(y_test, y_pred_bal, zero_division=0),
        f1_score(y_test, y_pred_bal, zero_division=0),
        roc_auc_score(y_test, y_proba_bal)
    ],
    "Tuned AdaBoost": [
        accuracy_score(y_test, best_pred),
        precision_score(y_test, best_pred, zero_division=0),
        recall_score(y_test, best_pred, zero_division=0),
        f1_score(y_test, best_pred, zero_division=0),
        roc_auc_score(y_test, best_proba)
    ]
}, index=["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"])

display(final_results.round(4))

## 20. Confusion Matrix

Confusion Matrix는 모델이 Non-Viral과 Viral을 각각 얼마나 잘 맞혔는지 보여줍니다.

특히 이 프로젝트에서는 모델이 Viral class를 놓치는지, 또는 Non-Viral을 Viral로 과하게 예측하는지 확인하는 데 중요합니다.

In [ ]:
cm = confusion_matrix(y_test, best_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Non-Viral", "Viral"]
)

fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False)
plt.title("Confusion Matrix — Tuned AdaBoost")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()

print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

## 21. ROC Curve

ROC Curve는 모델이 Viral과 Non-Viral을 얼마나 잘 구분하는지 보여줍니다.

ROC-AUC가 0.5에 가까우면 random guessing과 비슷한 수준이고, 1에 가까울수록 분류 성능이 좋다는 의미입니다.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, best_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AdaBoost (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Tuned AdaBoost")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 22. Feature Importance

AdaBoost는 feature importance를 제공하기 때문에, 어떤 변수가 모델 예측에 상대적으로 많이 사용되었는지 확인할 수 있습니다.

다만 전체 ROC-AUC가 낮다면, feature importance가 높게 나온 변수가 실제로 Viral 여부를 강하게 설명한다고 단정하기는 어렵습니다. 이 결과는 모델 내부에서 상대적으로 많이 사용된 feature를 보여주는 참고 자료로 해석해야 합니다.

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": best_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

display(feature_importance.head(15))

top_features = feature_importance.head(15).iloc[::-1]

plt.figure(figsize=(8, 6))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Feature Importance")
plt.title("Top 15 Feature Importances — AdaBoost")
plt.tight_layout()
plt.show()

# **XGBOOST**

 1. Library Import

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer


2. Random Seed

In [ ]:
RANDOM_STATE = 42


3. Load Dataset


In [ ]:
df = pd.read_csv('/content/instagram data.csv')

 4. Basic Data Check

In [ ]:
print("Dataset Shape:", df.shape)

print("\nFirst 5 Rows")
display(df.head())

print("\nData Info")
print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

5. Remove Leakage Features

In [ ]:
drop_columns = [
    'likes',
    'comments',
    'shares',
    'saves',
    'impressions',
    'reach',
    'engagement_rate',
    'reel_id',
    'creator_id'
]

# 존재하는 컬럼만 제거
df = df.drop(
    columns=[col for col in drop_columns if col in df.columns]
)

print("\nRemoved Leakage / ID Columns")


6. Create Target Label


In [ ]:
# Viral Score >= 80 -> Viral(1)
# Viral Score < 80 -> Non-Viral(0)
df['viral_label'] = (df['virality_score'] >= 80).astype(int)

# 기존 virality_score 제거
df = df.drop(columns=['virality_score'])

print("\nTarget Label Created")

7. Target Distribution

In [ ]:
print("\nTarget Distribution")
print(df['viral_label'].value_counts())

print("\nTarget Distribution Ratio")
print(df['viral_label'].value_counts(normalize=True))

# 시각화
plt.figure(figsize=(5,4))
sns.countplot(x='viral_label', data=df)
plt.title('Viral Label Distribution')
plt.show()

8. Separate X and y

In [ ]:
X = df.drop(columns=['viral_label'])
y = df['viral_label']

9. Identify Numeric / Categorical Columns

In [ ]:
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

print("\nNumeric Columns")
print(list(numeric_cols))

print("\nCategorical Columns")
print(list(categorical_cols))


 10. Missing Value Handling

In [ ]:
# Numeric -> Median Imputation
num_imputer = SimpleImputer(
    strategy='median'
)

X[numeric_cols] = num_imputer.fit_transform(
    X[numeric_cols]
)

# Categorical -> Most Frequent Imputation
if len(categorical_cols) > 0:

    cat_imputer = SimpleImputer(
        strategy='most_frequent'
    )

    X[categorical_cols] = cat_imputer.fit_transform(
        X[categorical_cols]
    )

else:
    print("\nNo categorical columns found.")

print("\nMissing Value Handling Complete")


11. One-Hot Encoding

In [ ]:
if len(categorical_cols) > 0:

    X = pd.get_dummies(
        X,
        columns=categorical_cols,
        drop_first=True
    )

else:
    print("\nNo categorical columns to encode.")

print("\nEncoding Complete")
print("Encoded Data Shape:", X.shape)

12. Correlation Heatmap

13. Numeric Feature Distribution

In [ ]:
for col in numeric_cols[:5]:

    plt.figure(figsize=(6,4))

    sns.histplot(df[col], kde=True)

    plt.title(f'Distribution of {col}')
    plt.show()

14. Boxplot for Outlier Check

In [ ]:
for col in numeric_cols[:5]:

    plt.figure(figsize=(6,4))

    sns.boxplot(x=df[col])

    plt.title(f'Boxplot of {col}')
    plt.show()

15. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print("\nTrain/Test Split Complete")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

16. Final Check

In [ ]:
print("\nTrain Label Distribution")
print(y_train.value_counts(normalize=True))

print("\nTest Label Distribution")
print(y_test.value_counts(normalize=True))

print("\nCommon Preprocessing & EDA Complete!")

# XG Boost

17. XG Boost modeling


In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# XGBoost 분류 모델 초기화
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    random_state=RANDOM_STATE,
    eval_metric='logloss',
    early_stopping_rounds=20
)

# 모델 학습
print("XGBoost 모델 학습을 시작합니다...")
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)
print("학습 완료!\n")

# 테스트 데이터로 예측 수행
y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

# 최종 성능 평가 지표 출력
print("=== XGBoost 모델 성능 평가 ===")
print(f"Accuracy (정확도): {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-Score (F1 점수): {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC 점수:      {roc_auc_score(y_test, y_pred_proba):.4f}")
print("\n[상세 분류 리포트]")
print(classification_report(y_test, y_pred))

# 피처 중요도 (Feature Importance) 시각화
plt.figure(figsize=(10, 6))
xgb.plot_importance(
    xgb_model,
    max_num_features=10,
    importance_type='gain',
    title='XGBoost Feature Importance (Top 10)',
    xlabel='Feature Importance (Gain)'
)
plt.show()

**18. 개선 1: scale_pos_weight=weight 코드로 가중치 부여**

In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# 1. XGBoost 분류 모델 초기화
weight = y_train.value_counts()[0] / y_train.value_counts()[1]
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    random_state=RANDOM_STATE,
    eval_metric='logloss',
    early_stopping_rounds=20,
    scale_pos_weight=weight
)

# 2. 모델 학습
print("XGBoost 모델 학습을 시작합니다.")
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

# 3. 테스트 데이터로 예측 수행
y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

# 4. 최종 성능 평가 지표 출력
print("=== XGBoost 모델 성능 평가 ===")
print(f"Accuracy (정확도): {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-Score (F1 점수): {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC 점수:      {roc_auc_score(y_test, y_pred_proba):.4f}")
print("\n[상세 분류 리포트]")
print(classification_report(y_test, y_pred))

# 5. 피처 중요도 (Feature Importance) 시각화
plt.figure(figsize=(10, 6))
xgb.plot_importance(
    xgb_model,
    max_num_features=10,
    importance_type='gain',
    title='XGBoost Feature Importance (Top 10)',
    xlabel='Feature Importance (Gain)'
)
plt.show()

**19. 개선 2 - hyperparameter tuning**

In [ ]:
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import matplotlib.pyplot as plt

print("원본 데이터 기반 하이퍼파라미터 튜닝을 시작합니다.\n")

# 튜닝할 하이퍼파라미터 설정
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200, 300]
}
weight = y_train.value_counts()[0] / y_train.value_counts()[1]

# 튜닝용 베이스 모델 생성
xgb_base = xgb.XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=weight
)

# 학습 시작
grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

# 튜닝 결과 확인 및 최적의 모델 추출
print("\n=== 하이퍼파라미터 튜닝 결과 ===")
print(f"교차 검증 중 최고 F1 점수: {grid_search.best_score_:.4f}")
print("찾아낸 최적의 조합:", grid_search.best_params_)

best_xgb_model = grid_search.best_estimator_


# 최종 Test 데이터 평가
y_pred_final = best_xgb_model.predict(X_test)
y_pred_proba_final = best_xgb_model.predict_proba(X_test)[:, 1]

print("\n=== 튜닝 완료 XGBoost 모델 성능 평가 ===")
print(f"Accuracy (정확도): {accuracy_score(y_test, y_pred_final):.4f}")
print(f"F1-Score (F1 점수): {f1_score(y_test, y_pred_final):.4f}")
print(f"ROC-AUC 점수:      {roc_auc_score(y_test, y_pred_proba_final):.4f}")
print("\n[상세 분류 리포트]")
print(classification_report(y_test, y_pred_final))

# 피처 중요도 시각화
plt.figure(figsize=(10, 6))
xgb.plot_importance(
    best_xgb_model,
    max_num_features=10,
    importance_type='gain',
    title='Tuned XGBoost Feature Importance (Original Features)',
    xlabel='Feature Importance (Gain)'
)
plt.show()